In [2]:
import pandas as pd
import sqlite3

In [3]:
df = pd.read_csv("client_site_dataset.csv")

In [4]:
df.head()

,User ID,Session ID,Event Time,Event,Device,Region,Channel,Product Category,Revenue,Bonus Flag
0,USR-00001,SES-00001,46:20.4,Browse,Tablet,South,Organic,Electronics,0.0,Yes
1,USR-00001,SES-00001,49:20.4,Add to Cart,Tablet,South,Organic,Electronics,0.0,Yes
2,USR-00001,SES-00001,51:20.4,Checkout,Tablet,South,Organic,Electronics,0.0,Yes
3,USR-00002,SES-00002,06:42.3,Browse,Desktop,East,Email,Sports,0.0,Yes
4,USR-00002,SES-00002,10:42.3,Add to Cart,Desktop,East,Email,Sports,0.0,Yes


In [5]:
print(df.shape)

(21409, 10)


In [6]:
print(df.columns.tolist())

['User ID', 'Session ID', 'Event Time', 'Event', 'Device', 'Region', 'Channel', 'Product Category', 'Revenue', 'Bonus Flag']


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21409 entries, 0 to 21408
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   User ID           21409 non-null  str    
 1   Session ID        21409 non-null  str    
 2   Event Time        21409 non-null  str    
 3   Event             21409 non-null  str    
 4   Device            21409 non-null  str    
 5   Region            21409 non-null  str    
 6   Channel           21409 non-null  str    
 7   Product Category  21409 non-null  str    
 8   Revenue           21409 non-null  float64
 9   Bonus Flag        21409 non-null  str    
dtypes: float64(1), str(9)
memory usage: 1.6 MB


In [8]:
df.isnull().sum()

User ID             0
Session ID          0
Event Time          0
Event               0
Device              0
Region              0
Channel             0
Product Category    0
Revenue             0
Bonus Flag          0
dtype: int64

In [9]:
print(df.duplicated().sum())

0


In [10]:
df.describe()

,Revenue
count,21409.000000
mean,12.953574
std,65.005620
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,499.990000


In [3]:
conn = sqlite3.connect("client_site.db")

In [4]:
df.to_sql(
    "client_data",
    conn,
    if_exists = "replace",
    index = False
)

21409

In [13]:
query = """
SELECT *
FROM client_data
LIMIT 5;
"""

pd.read_sql(query, conn)

,User ID,Session ID,Event Time,Event,Device,Region,Channel,Product Category,Revenue,Bonus Flag
0,USR-00001,SES-00001,46:20.4,Browse,Tablet,South,Organic,Electronics,0.0,Yes
1,USR-00001,SES-00001,49:20.4,Add to Cart,Tablet,South,Organic,Electronics,0.0,Yes
2,USR-00001,SES-00001,51:20.4,Checkout,Tablet,South,Organic,Electronics,0.0,Yes
3,USR-00002,SES-00002,06:42.3,Browse,Desktop,East,Email,Sports,0.0,Yes
4,USR-00002,SES-00002,10:42.3,Add to Cart,Desktop,East,Email,Sports,0.0,Yes


In [22]:
# Total number of records
query = """
SELECT COUNT(*) AS Total_Records
FROM client_data;
"""

pd.read_sql(query, conn)

,Total_Records
0,21409


In [16]:
query = """
SELECT COUNT(DISTINCT User_ID) AS Total_Users
FROM client_data;
"""

pd.read_sql(query, conn)

DatabaseError: Execution failed on sql '
SELECT COUNT(DISTINCT User_ID) AS Total_Users
FROM client_data;
': no such column: User_ID

In [17]:
print(df.columns.tolist())

['User ID', 'Session ID', 'Event Time', 'Event', 'Device', 'Region', 'Channel', 'Product Category', 'Revenue', 'Bonus Flag']


In [23]:
# Total Unique Users
query = """
SELECT COUNT(DISTINCT "User ID") AS Total_Users
FROM client_data;
"""

pd.read_sql(query, conn)

,Total_Users
0,10000


In [24]:
# Total unique sessions
query = """
SELECT COUNT(DISTINCT "Session ID") AS Total_Sessions
FROM client_data;
"""

pd.read_sql(query, conn)

,Total_Sessions
0,10000


In [25]:
# Different types of user events
query = """
SELECT DISTINCT Event
FROM client_data;
"""

pd.read_sql(query, conn)

,Event
0,Browse
1,Add to Cart
2,Checkout
3,Purchase


In [26]:
# Task 2 Funnel Stage Analysis

In [27]:
# Total events by Type
query = """
SELECT 
    Event,
    Count(*) AS Total_Events
FROM client_data
GROUP BY Event
ORDER BY Total_Events DESC;
"""

pd.read_sql(query, conn)

,Event,Total_Events
0,Browse,10000
1,Add to Cart,6949
2,Checkout,3456
3,Purchase,1004


In [28]:
# Users per Event Type
query = """
SELECT
    Event,
    COUNT(DISTINCT "User ID") AS Total_Users
FROM client_data
GROUP BY Event
ORDER BY Total_Users DESC;
"""

pd.read_sql(query, conn)

,Event,Total_Users
0,Browse,10000
1,Add to Cart,6949
2,Checkout,3456
3,Purchase,1004


In [37]:
# Conversion from View -> Purchase
# Users who viewed
query = """
SELECT COUNT(DISTINCT "User ID") AS Browse_Users
FROM client_data
WHERE Event = 'Browse';
"""

browse_df = pd.read_sql(query, conn)
browse_df

,Browse_Users
0,10000


In [38]:
Browse_Users = browse_df.loc[0, "Browse_Users"]
print(Browse_Users)

10000


In [39]:
# Users who purchased
query = """
SELECT COUNT(DISTINCT "User ID") AS Purchase_Users
FROM client_data
WHERE Event = 'Purchase';
"""

purchase_df = pd.read_sql(query, conn)
purchase_df

,Purchase_Users
0,1004


In [40]:
Purchase_Users = purchase_df.loc[0, "Purchase_Users"]
print(Purchase_Users)

1004


In [44]:
# Conversion Rate
Conversion_Rate = (Purchase_Users / Browse_Users) * 100

print(f"Conversion_Rate: {Conversion_Rate: .2f}%")

Conversion_Rate:  10.04%


In [45]:
# Task 3 Revenue Analysis


In [55]:
# Total Revenue
query = """
SELECT SUM(Revenue) AS Total_Revenue
FROM client_data
WHERE Event = 'Purchase';
"""

pd.read_sql(query, conn)

,Total_Revenue
0,277323.06


In [56]:
# Revenue by Region
query = """
SELECT
    Region,
    SUM(Revenue) AS Total_Revenue
FROM client_data
WHERE Event = 'Purchase'
GROUP BY Region
ORDER BY Total_Revenue DESC;
"""

pd.read_sql(query, conn)

,Region,Total_Revenue
0,South,77421.45
1,North,68645.13
2,East,66116.01
3,West,65140.47


In [57]:
# Revenue by Channel
query = """
SELECT
    Channel,
    SUM(Revenue) AS Total_Revenue
FROM client_data
WHERE Event = 'Purchase'
GROUP BY Channel
ORDER BY Total_Revenue DESC;
"""

pd.read_sql(query, conn)

,Channel,Total_Revenue
0,Google Ads,73862.32
1,Email,69126.46
2,Social Media,68361.24
3,Organic,65973.04


In [58]:
# Revenue by Device
query = """
SELECT
    Device,
    SUM(Revenue) AS Total_Revenue
FROM client_data
WHERE Event = 'Purchase'
GROUP BY Device
ORDER BY Total_Revenue DESC;
"""

pd.read_sql(query, conn)

,Device,Total_Revenue
0,Desktop,98471.83
1,Tablet,94620.13
2,Mobile,84231.10


In [6]:
# Task 4 Business Insights Queries

In [7]:
# Top 5 Users by Revenue
query = """
SELECT
    "USER ID",
    SUM(Revenue) AS Total_Revenue
FROM client_data
WHERE EVENT = 'Purchase'
GROUP BY "User ID"
ORDER BY Total_Revenue DESC
LIMIT 5;
"""

pd.read_sql(query, conn)

,User ID,Total_Revenue
0,USR-04835,499.99
1,USR-00990,499.64
2,USR-04088,499.15
3,USR-08031,498.96
4,USR-05780,498.86


In [8]:
# Best performing Channel
query = """
SELECT 
    Channel,
    SUM(Revenue) AS Total_Revenue
FROM client_data
WHERE Event = 'Purchase'
GROUP BY Channel
ORDER BY Total_Revenue DESC
LIMIT 1;
"""

pd.read_sql(query, conn)


,Channel,Total_Revenue
0,Google Ads,73862.32


In [9]:
# Highest Revenue Region
query = """
SELECT
    Region,
    SUM(Revenue) AS Total_Revenue
FROM client_data
WHERE Event = 'Purchase'
GROUP BY Region
ORDER BY Total_Revenue DESC
LIMIT 1;
"""

pd.read_sql(query, conn)

,Region,Total_Revenue
0,South,77421.45


In [12]:
query = """
SELECT
    Device,
    SUM(CASE WHEN Event = 'Purchase' THEN 1 ELSE 0 END) AS Purchases,
    SUM(CASE WHEN Event = 'Browse' THEN 1 ELSE 0 END) AS Browses
FROM client_data
GROUP BY Device;
"""

pd.read_sql(query, conn)

,Device,Purchases,Browses
0,Desktop,356,3366
1,Mobile,309,3263
2,Tablet,339,3371


In [13]:
result = pd.read_sql(query, conn)

result["Conversion Rate"] = (
    result["Purchases"] / result["Browses"]
) * 100

result.sort_values("Conversion Rate", ascending=False)

,Device,Purchases,Browses,Conversion Rate
0,Desktop,356,3366,10.576352
2,Tablet,339,3371,10.056363
1,Mobile,309,3263,9.469813


In [14]:
query = """
SELECT DISTINCT Device
FROM client_data;
"""

pd.read_sql(query, conn)

,Device
0,Tablet
1,Desktop
2,Mobile


In [1]:
# Task 5 Drop-off Analysis

In [7]:
query = """
SELECT
    Event,
    COUNT(DISTINCT "User ID") AS Users
FROM client_data
GROUP BY Event;
"""

pd.read_sql(query, conn)

,Event,Users
0,Add to Cart,6949
1,Browse,10000
2,Checkout,3456
3,Purchase,1004


In [8]:
query = """
SELECT 
    Event,
    COUNT(DISTINCT "User ID") AS Users
FROM client_data
GROUP BY Event;
"""

funnel = pd.read_sql(query, conn)

funnel

,Event,Users
0,Add to Cart,6949
1,Browse,10000
2,Checkout,3456
3,Purchase,1004


In [12]:
browse = funnel.loc[funnel["Event"] == "Browse", "Users"].values[0]
cart = funnel.loc[funnel["Event"] == "Add to Cart", "Users"].values[0]
checkout = funnel.loc[funnel["Event"] == "Checkout", "Users"].values[0]
purchase = funnel.loc[funnel["Event"] == "Checkout", "Users"].values[0]

browse_to_cart = (cart / browse) * 100
cart_to_checkout = (checkout / cart) * 100
checkout_to_purchase = (purchase / checkout) * 100

print(f"Browse -> Add to Cart: {browse_to_cart: .2f}%")
print(f"Add to Cart -> Checkout: {cart_to_checkout: .2f}%")
print(f"Checkout -> Purchase: {checkout_to_purchase: .2f}%")

Browse -> Add to Cart:  69.49%
Add to Cart -> Checkout:  49.73%
Checkout -> Purchase:  100.00%


In [6]:
df.to_csv("client_data_cleaned.csv", index=False)

In [7]:

df["Event Time"].head(10)

0    46:20.4
1    49:20.4
2    51:20.4
3    06:42.3
4    10:42.3
5    49:00.9
6    58:00.8
7    00:00.8
8    02:00.8
9    56:31.7
Name: Event Time, dtype: str